**1. Load the Twitter dataset**

In [0]:
twitter_path = "/Volumes/workspace/retail_lab/raw_files/tweets.avro"

twitter_df = spark.read.format("avro").load(twitter_path)

display(twitter_df.limit(10))

tweet_id,created_at,text,lang,source,is_retweet,reply_to_tweet_id,user,hashtags,mentions,metrics,geo
9000000000,2026-10-12T20:17:15.000Z,RT Thread on why #Travel #Music #Movies matters more than ever @aarav_muller101,en,Twitter Web App,true,null,"List(1001, ingrid_gupta1, Ingrid Gupta, false, 105, 75, US, 2026-03-24T00:00:00.000Z)","List(Travel, Music, Movies)",List(aarav_muller101),"List(0, 0, 0, 0)","List(Austin, 30.23235887367635, -97.75314143413839)"
9000000001,2026-10-28T22:20:49.000Z,Thread on why this matters more than ever,en,Twitter Web App,false,null,"List(1001, ingrid_gupta1, Ingrid Gupta, false, 105, 75, US, 2026-03-24T00:00:00.000Z)",List(),List(),"List(2, 0, 0, 0)",null
9000000002,2026-10-05T18:37:49.000Z,Anyone else following this today?,fr,Twitter for Android,false,null,"List(1197, emma_tanaka197, Emma Tanaka, false, 56, 1114, FR, 2016-09-25T00:00:00.000Z)",List(),List(),"List(3, 0, 0, 0)",null
9000000003,2026-10-14T09:26:25.000Z,Morning coffee and some this reading @rohan_kim55,en,TweetDeck,false,null,"List(1161, mei_singh161, Mei Singh, false, 164, 426, NG, 2023-07-27T00:00:00.000Z)",List(),List(rohan_kim55),"List(3, 1, 0, 0)","List(Lagos, 6.524446586148214, 3.4277842529452047)"
9000000004,2026-10-25T21:17:17.000Z,Just tried this and I'm impressed! @ananya_gupta5,en,Twitter for Android,false,null,"List(1000, aarav_khan0, Aarav Khan, false, 63, 295, US, 2018-04-21T00:00:00.000Z)",List(),List(ananya_gupta5),"List(1, 0, 0, 0)",null
9000000005,2026-10-26T15:55:36.000Z,Anyone else following #Cricket today? @noah_smith37,en,Twitter for iPhone,false,null,"List(1090, ananya_kumar90, Ananya Kumar, false, 61, 1656, IN, 2018-10-04T00:00:00.000Z)",List(Cricket),List(noah_smith37),"List(11, 3, 1, 0)",null
9000000006,2026-10-14T10:11:39.000Z,Hot take: this is overrated.,de,Twitter for Android,false,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)",List(),List(),"List(5, 0, 0, 0)",null
9000000007,2026-10-26T18:12:56.000Z,Morning coffee and some #AI reading,en,Twitter for iPhone,false,null,"List(1026, mateo_wang26, Mateo Wang, false, 62, 503, US, 2016-09-18T00:00:00.000Z)",List(AI),List(),"List(3, 0, 0, 0)",null
9000000008,2026-10-27T04:26:56.000Z,Big news around #Movies right now,en,Twitter for Android,false,null,"List(1043, rohan_sato43, Rohan Sato, false, 58, 1004, IN, 2021-03-01T00:00:00.000Z)",List(Movies),List(),"List(6, 2, 0, 0)",null
9000000009,2026-10-08T23:26:17.000Z,Big news around #Worldcup #Python right now,en,Twitter for Android,false,null,"List(1013, olivia_nielsen13, Olivia Nielsen, false, 233, 479, CA, 2017-05-06T00:00:00.000Z)","List(Worldcup, Python)",List(),"List(2, 0, 0, 0)","List(Toronto, 43.609323010170364, -79.42034355674342)"


**2. Check the schema**

In [0]:
twitter_df.printSchema()

root
 |-- tweet_id: long (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- text: string (nullable = true)
 |-- lang: string (nullable = true)
 |-- source: string (nullable = true)
 |-- is_retweet: boolean (nullable = true)
 |-- reply_to_tweet_id: long (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- user_id: long (nullable = true)
 |    |-- username: string (nullable = true)
 |    |-- display_name: string (nullable = true)
 |    |-- verified: boolean (nullable = true)
 |    |-- followers_count: integer (nullable = true)
 |    |-- following_count: integer (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- account_created: timestamp (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- mentions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- metrics: struct (nullable = true)
 |    |-- likes: integer (nullable = true)
 |    |-- retweets: integer (null

**3. Count the tweets**

In [0]:
print("Total tweets:", twitter_df.count())

Total tweets: 3000


**4. Identify the top 5 users with the highest number of tweets**

In [0]:
from pyspark.sql.functions import count, col

top_5_users = twitter_df \
    .groupBy(col("user.username").alias("username")) \
    .agg(count("*").alias("tweet_count")) \
    .orderBy(col("tweet_count").desc()) \
    .limit(5)

display(top_5_users)

username,tweet_count
aarav_khan0,304
ingrid_gupta1,187
noah_sato2,134
ingrid_kumar3,111
emma_kumar4,70


**2. Most commonly used words in all the tweets, excluding stopwords**


2.1. Split the tweets into individual words

In [0]:
from pyspark.sql.functions import explode, split, lower, col

words_df = twitter_df.select(
    explode(split(lower(col("text")), " ")).alias("word")
)

display(words_df)

word
rt
thread
on
why
#travel
#music
#movies
matters
more
than


2.2. List common words to exclude

In [0]:
stopwords = [
    "the", "is", "a", "an", "and", "or", "to",
    "of", "in", "for", "on", "with", "at", "by",
    "this", "that", "it", "are", "was", "be",
    "i", "you", "we", "they", "he", "she"
]

2.3.Remove stopwords and empty words

words_df: The DataFrame containing all individual words from the tweets.



.filter(): Selects only rows that meet a condition.



col("word"): Refers to the column named word.



!= "": Means is not empty.

In [0]:
from pyspark.sql.functions import col, explode, split, lower, count

filtered_words = words_df.filter(col("word") != "")

filtered_words = filtered_words.filter(
    ~col("word").isin(stopwords)
)

2.4. Count and display the most common words

In [0]:
word_counts = filtered_words.groupBy("word") \
    .count() \
    .orderBy(col("count").desc())

display(word_counts.limit(20))

word,count
rt,457
overrated.,403
hot,403
take:,403
can't,399
about,399
thinking,399
stop,399
following,390
today?,390


# **3. Filter the tweets that are in Spanish**

twitter_df: Our original tweets DataFrame.



filter(): Selects rows matching a condition.



col("lang"): Refers to the language column.



== "es": Selects tweets where the language is Spanish.

In [0]:
spanish_tweets = twitter_df.filter(col("lang") == "es")

display(spanish_tweets)

tweet_id,created_at,text,lang,source,is_retweet,reply_to_tweet_id,user,hashtags,mentions,metrics,geo


In [0]:
print(spanish_tweets.count())

0


In [0]:
twitter_df.groupBy("lang") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+----+-----+
|lang|count|
+----+-----+
|  en| 2144|
|  ja|  309|
|  de|  249|
|  pt|  179|
|  fr|  119|
+----+-----+



**4. Which country has the highest number of tweets?**

user.country: Gets the country from the user's profile.



groupBy(): Groups tweets by country.



count("*"): Counts tweets for each country.



orderBy(): Sorts countries from highest to lowest tweet count.



display(): Shows the results.

In [0]:
from pyspark.sql.functions import col, count

tweets_by_country = twitter_df \
    .groupBy(col("user.country").alias("country")) \
    .agg(count("*").alias("total_tweets")) \
    .orderBy(col("total_tweets").desc())

display(tweets_by_country)

country,total_tweets
US,859
CA,362
JP,333
DE,274
IN,259
AU,256
NG,221
BR,196
FR,135
GB,105


In [0]:
display(tweets_by_country.limit(1))

country,total_tweets
US,859


**5. Separate tweets by a specific username**

user.username: Refers to the username inside the user structure.



filter(): Selects only tweets matching the condition.



== "Mark": Keeps tweets from the username Mark.



display(): Shows the filtered tweets.

In [0]:
mark_tweets = twitter_df.filter(col("user.username") == "Mark")

display(mark_tweets)

tweet_id,created_at,text,lang,source,is_retweet,reply_to_tweet_id,user,hashtags,mentions,metrics,geo


In [0]:
print("Total tweets:", mark_tweets.count())

Total tweets: 0


In [0]:
twitter_df.select("user.username").distinct().show(20, False)

+----------------+
|username        |
+----------------+
|ananya_gupta5   |
|ingrid_sharma164|
|mateo_garcia10  |
|ingrid_patel64  |
|zara_nielsen93  |
|yuki_kim21      |
|aarav_gupta33   |
|aarav_brown124  |
|zara_nielsen79  |
|ravi_muller106  |
|olivia_brown185 |
|noah_smith37    |
|hiro_rossi102   |
|hiro_lee24      |
|ananya_rossi7   |
|zara_gupta20    |
|omar_patel100   |
|noah_gupta99    |
|sofia_gupta86   |
|chen_kim167     |
+----------------+
only showing top 20 rows


In [0]:
username_tweets = twitter_df.filter(
    col("user.username") == "ananya_gupta5"
)

display(username_tweets)

print("Total tweets:", username_tweets.count())

tweet_id,created_at,text,lang,source,is_retweet,reply_to_tweet_id,user,hashtags,mentions,metrics,geo
9000000006,2026-10-14T10:11:39.000Z,Hot take: this is overrated.,de,Twitter for Android,false,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)",List(),List(),"List(5, 0, 0, 0)",null
9000000037,2026-10-28T09:43:24.000Z,Morning coffee and some #Startup reading,de,TweetDeck,false,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)",List(Startup),List(),"List(1, 0, 0, 0)",null
9000000099,2026-10-27T11:28:56.000Z,Just tried #Worldcup #MachineLearning #Travel and I'm impressed!,de,Twitter Web App,false,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)","List(Worldcup, MachineLearning, Travel)",List(),"List(15, 4, 1, 1)",null
9000000187,2026-10-29T17:53:48.000Z,Thread on why this matters more than ever @ananya_kim132,de,Twitter Web App,false,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)",List(),List(ananya_kim132),"List(19, 2, 2, 0)","List(Munich, 48.10032250128965, 11.626844140083621)"
9000000256,2026-10-15T01:21:45.000Z,RT Anyone else following #Travel today? @emma_singh12,de,Twitter for iPhone,true,9000000213,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)",List(Travel),List(emma_singh12),"List(0, 0, 0, 0)","List(Berlin, 52.53669965429462, 13.433801443579261)"
9000000343,2026-10-27T22:12:54.000Z,Morning coffee and some #Music reading @emma_garcia184,de,Twitter Web App,false,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)",List(Music),List(emma_garcia184),"List(0, 0, 0, 0)","List(Munich, 48.176717177974865, 11.620057488497412)"
9000000354,2026-10-09T05:42:48.000Z,RT Morning coffee and some #BigData reading,de,TweetDeck,true,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)",List(BigData),List(),"List(0, 0, 0, 0)","List(Munich, 48.187439917693304, 11.569755471562484)"
9000000372,2026-10-11T19:20:36.000Z,"Learning #Music #MachineLearning this weekend, wish me luck",de,Twitter for Android,false,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)","List(Music, MachineLearning)",List(),"List(5, 0, 0, 0)",null
9000000489,2026-10-06T03:37:40.000Z,"Learning #Cricket #AI #Startup this weekend, wish me luck",en,Twitter for Android,false,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)","List(Cricket, AI, Startup)",List(),"List(4, 0, 0, 0)",null
9000000584,2026-10-04T09:43:06.000Z,RT Just tried this and I'm impressed!,de,Twitter for Android,true,null,"List(1005, ananya_gupta5, Ananya Gupta, false, 74, 745, DE, 2017-01-21T00:00:00.000Z)",List(),List(),"List(0, 0, 0, 0)",null


Total tweets: 68
